### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [85]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [86]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF Files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")


Found 3 PDF Files to process

Processing: ai_data_science_convergence.pdf
 Loaded 1 pages

Processing: essentials_of_data_science.pdf
 Loaded 1 pages

Processing: fundamentals_of_ai.pdf
 Loaded 1 pages

Total documents loaded: 3


In [87]:
all_pdf_documents

[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'The Convergence of AI and Data Science', 'source': '..\\data\\pdf\\ai_data_science_convergence.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_data_science_convergence.pdf', 'file_type': 'pdf'}, page_content='The Convergence of AI and Data Science\nSynergies, MLOps, and the Future of Enterprise Intelligence\n1. The Synergy Between AI and Data Science\nWhile Data Science focuses on formulating queries, structuring datasets, and analyzing trends,\nArtificial Intelligence provides automated decision-making engines. Together , they enable automated\nend-to-end data pipelines and intelligent analytics platforms.\n2. Modern MLOps & Infrastructure\nMachine Learning Operations (MLOps) bridges the gap between model training and production\ndeployment, enforcing continuous integration, monitoring, and automated retraining.\nFeature Stores: Centralized repositories for man

In [88]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        seperators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [89]:
# Fix the typo in the splitter configuration (cell 4)
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    return split_docs

# Then create the chunks
chunks = split_documents(all_pdf_documents)
chunks

Split 3 documents into 7 chunks


[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'The Convergence of AI and Data Science', 'source': '..\\data\\pdf\\ai_data_science_convergence.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_data_science_convergence.pdf', 'file_type': 'pdf'}, page_content='The Convergence of AI and Data Science\nSynergies, MLOps, and the Future of Enterprise Intelligence\n1. The Synergy Between AI and Data Science\nWhile Data Science focuses on formulating queries, structuring datasets, and analyzing trends,\nArtificial Intelligence provides automated decision-making engines. Together , they enable automated\nend-to-end data pipelines and intelligent analytics platforms.\n2. Modern MLOps & Infrastructure\nMachine Learning Operations (MLOps) bridges the gap between model training and production\ndeployment, enforcing continuous integration, monitoring, and automated retraining.\nFeature Stores: Centralized repositories for man

### Embedding and VectorStoreDB

In [90]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import hashlib
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [91]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """

        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embeddding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """

        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape(len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embeddding dimension: 384


C:\Users\Adwait\AppData\Local\Temp\ipykernel_22896\3115126673.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embeddding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [92]:
class VectorStore:
    """Manages document embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents_v2", persistent_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persistent_directory: Directory to persistent the vector store
        """

        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB Client
            os.makedirs(self.persistent_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a stable ID so rerunning ingestion updates instead of duplicating chunks.

            identity = f"{doc.metadata.get('source_file', '')}:{doc.metadata.get('page', i)}:{doc.page_content}"
            doc_id = f"doc_{hashlib.sha256(identity.encode('utf-8')).hexdigest()}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection

        try:
            self.collection.upsert(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents_v2
Existing documents in collection: 7


In [93]:
chunks

[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'The Convergence of AI and Data Science', 'source': '..\\data\\pdf\\ai_data_science_convergence.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_data_science_convergence.pdf', 'file_type': 'pdf'}, page_content='The Convergence of AI and Data Science\nSynergies, MLOps, and the Future of Enterprise Intelligence\n1. The Synergy Between AI and Data Science\nWhile Data Science focuses on formulating queries, structuring datasets, and analyzing trends,\nArtificial Intelligence provides automated decision-making engines. Together , they enable automated\nend-to-end data pipelines and intelligent analytics platforms.\n2. Modern MLOps & Infrastructure\nMachine Learning Operations (MLOps) bridges the gap between model training and production\ndeployment, enforcing continuous integration, monitoring, and automated retraining.\nFeature Stores: Centralized repositories for man

In [94]:
### Convert the text into embeddings
texts = [doc.page_content for doc in chunks]
texts


## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 7 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (7, 384)
Adding 7 documents to vector store...
Successfully added 7 documents to vector store
Total documents in collection: 7


### Retriever Pipeline from VectorStore

In [95]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieiving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1.0 / (1.0 + max(float(distance), 0.0))

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore, embedding_manager)

In [96]:
rag_retriever

In [97]:
rag_retriever.retrieve("The Synergy Between AI and Data Science")

Retrieiving documents for query: 'The Synergy Between AI and Data Science'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c18cc8041037312a009f1e3fa25d9c5e031379718c0bf1c2a54fce4a3c7f3f83',
  'content': 'The Convergence of AI and Data Science\nSynergies, MLOps, and the Future of Enterprise Intelligence\n1. The Synergy Between AI and Data Science\nWhile Data Science focuses on formulating queries, structuring datasets, and analyzing trends,\nArtificial Intelligence provides automated decision-making engines. Together , they enable automated\nend-to-end data pipelines and intelligent analytics platforms.\n2. Modern MLOps & Infrastructure\nMachine Learning Operations (MLOps) bridges the gap between model training and production\ndeployment, enforcing continuous integration, monitoring, and automated retraining.\nFeature Stores: Centralized repositories for managing and serving features for training and\ninference.\nModel Monitoring: Continuous evaluation of data drift, concept drift, and performance metrics.\nAutomated Pipelines: CI/CD workflows tailored to machine learning models.\n3. Emerging H

In [98]:
rag_retriever.retrieve(" The Data Science Lifecycle")

Retrieiving documents for query: ' The Data Science Lifecycle'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_2234f098ff1ce294eab549c59ee5e11bf80352ff01f671219cb2c3347a362f40',
  'content': 'Essentials of Data Science\nMethods, Pipelines, and Analytics Frameworks\n1. The Data Science Lifecycle\nData Science is an interdisciplinary domain combining statistical methodology, computational thinking,\nand domain expertise to extract actionable insights from structured and unstructured data.\nPhase Primary Objective Key Techniques / Tools\nData Ingestion\nCollect raw data across databases, APIs,\nand streams\nSQL, Web Scraping, Kafka, ETL\nPipelines\nData Wrangling Clean, transform, and handle missing\nattributes\nPandas, NumPy, Imputation,\nNormalization\nExploratory Data\nAnalysis\nDiscover underlying distributions and\nrelationships\nSummary Statistics, Matplotlib,\nSeaborn\nPredictive Modeling Build and validate statistical/ML models\nScikit-learn, Cross-validation,\nRegression\n2. Key Statistical Foundations\nRigorous inference requires a solid mathematical foundation:\nDescriptive

### RAG Pipeline - VectorDB to LLM Output Generation

In [99]:
import os
from dotenv import load_dotenv
load_dotenv()

print("GROQ_API_KEY is configured:", bool(os.getenv("GROQ_API_KEY")))

GROQ_API_KEY is configured: True


In [100]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [101]:
class GroqLLM:
    def __init__(self, model_name: str= "openai/gpt-oss-120b", api_key: str=None):
        """
        Initialize Groq LLM

        Args:
            model_name: Groq model ID, for example openai/gpt-oss-120b
            api_key: Groq API Key (or set GROQ_API_KEY environment variable)
        """

        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API Key is required. Set GROQ_API_KEY environment variable or pass api_key_parameter.")

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """

        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables = ["context", "question"],
            template = """You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
            
        Context:
         {context}

        Question: {question}

        Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so.

            """
        )

        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated Response
        """

        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error: {str(e)}"

In [102]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment_variable)

try: 
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")

except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Groq LLM initialized successfully!


In [103]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Core Paradigms of AI")

Retrieiving documents for query: 'Core Paradigms of AI'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_fde428c264560348c9e01a413d1810aa0f197d8ce3f4b9b7aa6f3b477730e16b',
  'content': 'Fundamentals of Artificial Intelligence\nA Comprehensive Guide to AI Paradigms, Architectures, and Applications\n1. Introduction to Artificial Intelligence\nArtificial Intelligence (AI) refers to the simulation of human intelligence in machines programmed to think,\nlearn, and problem-solve. Modern AI encompasses various subfields, ranging from rule-based expert systems\nto complex neural architectures capable of processing unstructured data.\nKey Milestone: The shift from symbolic AI (rule-based logic) to statistical machine learning enabled\nautomated pattern extraction from massive datasets, revolutionizing perception tasks such as computer\nvision and natural language processing. \n2. Core Paradigms of AI\nMachine Learning (ML): Algorithms that optimize performance on tasks using empirical data without\nexplicit programmed rules.\nDeep Learning (DL): A subset of ML utilizing multi-layered 

### Integration Vectordb context pipeline with LLM output

In [104]:
### Simple RAG Pipeline with Groq LLM

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Check your .env file"
    )

llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using groq LLM
    prompt = f"""Use the following context to answer the question concisely.
    
        Context:
        {context}

        Question: {query}

        Answer: """

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [105]:
from IPython.display import Markdown, display

answer = rag_simple("Neural Network Architectures", rag_retriever, llm)
display(Markdown(answer))

Retrieiving documents for query: 'Neural Network Architectures'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Neural Network Architectures**

- **Convolutional Neural Networks (CNNs)** – Optimized for spatial grid data such as images and video; they use convolutional filters to capture local patterns.  
- **Recurrent Neural Networks (RNNs) & Long Short‑Term Memory networks (LSTMs)** – Designed for sequential or temporal data (e.g., speech, time series); they maintain hidden states to model dependencies over time.  
- **Transformers** – Built on self‑attention mechanisms that process sequence elements in parallel; they form the backbone of modern Large Language Models (LLMs) and excel at handling long‑range dependencies.

### Enhanced RAG Pipeline Features

In [106]:
# -- Enhanced RAG Pipeline Features ---

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG Pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.

    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output

result = rag_advanced("Hard Negative Mining Techniques", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieiving documents for query: 'Hard Negative Mining Techniques'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: **Hard Negative Mining (HNM)** is a training‑time strategy that deliberately selects the most challenging “negative” (i.e., non‑target) examples for a model to learn from, rather than using all negatives or random samples. By focusing on these difficult cases, the model improves its discriminative power and reduces false‑positive errors.

### Why It’s Used
- **Class imbalance:** In many vision or retrieval tasks, true positives are far fewer than negatives. Random sampling would waste capacity on easy negatives that contribute little to learning.
- **Model robustness:** Exposing the model to hard negatives forces it to refine decision boundaries, leading to higher precision and better generalization.

### Common Techniques  

| Technique | Core Idea | Typical Workflow |
|-----------|----------|------------------|
| **Offline (pre‑computed) mining** | Rank all negatives by a loss‑or confidence scor

In [107]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found"
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            #Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely. \nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer

        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return{
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Key Statistical Foundations", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieiving documents for query: 'Key Statistical Foundations'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely. 
Context:
Regression
2. Key Statistical Foundations
Rigorous inference requires a solid mathematical foundation:
Descriptive Statistics: Measures of central tendency, variance, skewness, and correlation.
Inferential Statistics: Hypothesis testing (p-values, t-tests, ANOVA) and confidence intervals.
Probability Distributions: Gaussian, Binomial, and Poisson models governing real-world signals.
3. Machine Learning in Data Science
Data scientists apply supervised learning for classification and regression, and unsupervised learning
(e.g., K-Means, PCA) for clustering and dimensionality reduction to uncover hidden structures in
empirical data.
Data Science Knowledge Series • Reference Document #2 
• 
• 
•

Essentials of Data Science
Methods, Pipelines, and Analytics Frameworks
1. The Data Science Lifecycle
Data Science is an interdi